# Cloud RAG Evaluation Notebook

This notebook evaluates the performance of a Retrieval-Augmented Generation (RAG) system built on multi-cloud documentation (AWS, Azure, GCP).

The evaluation focuses on assessing:
- Answer correctness
- Groundedness (faithfulness to retrieved context)
- Completeness of responses
- Topic and provider coverage

The RAG pipeline is implemented using a modular architecture with:
- Vector database (ChromaDB)
- Embedding model (SentenceTransformers)
- Retrieval + optional reranking
- LLM-based answer generation (OpenAI)

---

## ⚙️ Evaluation Methodology

The evaluation combines:

1. **Heuristic Metrics**
   - Topic coverage
   - Provider coverage
   - Semantic grounding checks


2. **LLM-as-a-Judge**
   - Correctness
   - Relevance
   - Completeness
   - Groundedness

All evaluation logic is centralized in reusable modules to ensure consistency across experiments.

---

## 📌 Note

Due to large dataset size (~75k chunks), the vector database and raw documents are not included in the repository.  
The system is fully reproducible using the provided data ingestion and embedding pipeline.

**Skript flow diagram**

In [1]:
import os
import sys
import re
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

#PROJECT_ROOT = Path(r"E:/GenAi/project/Rag_cloud_platforms")
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Load environment variables
load_dotenv(PROJECT_ROOT / "src" / ".env")
print("OPENAI_API_KEY loaded?:", bool(os.getenv("OPENAI_API_KEY")))

from src.settings import CHROMA_DIR, COLLECTION_NAME
from src.vectorstore import get_retriever
from src.rag_lcel import build_chain

OPENAI_API_KEY loaded?: True


### Initialize RAG Chain

In [2]:
chain = build_chain(where=None)

def run_query(query):
    return chain.invoke(query)

### Test Queries

In [3]:
aws_questions = [
    {
        "query": "What is AWS Well Architected Framework?",
        "expected_keywords": ["reliability", "security", "cost"],
        "expected_providers": ["aws"],
        "topic": "well-architected",
        "must_contain": ["pillars"],
        "must_not_contain": ["Azure", "GCP"]
    },
    {
        "query": "Explain AWS IAM roles and policies",
        "expected_keywords": ["roles", "policies", "permissions"],
        "expected_providers": ["aws"],
        "topic": "iam",
        "must_contain": ["roles", "policies"],
        "must_not_contain": ["Azure RBAC"]
    },
    {
        "query": "What is AWS Control Tower?",
        "expected_keywords": ["multi-account", "governance"],
        "expected_providers": ["aws"],
        "topic": "landing-zone",
        "must_contain": ["accounts"],
        "must_not_contain": ["GCP projects"]
    },
    {
        "query": "Explain AWS VPC components",
        "expected_keywords": ["subnet", "route", "gateway"],
        "expected_providers": ["aws"],
        "topic": "networking",
        "must_contain": ["subnet"],
        "must_not_contain": ["Azure VNet"]
    },
    {
        "query": "What is AWS shared responsibility model?",
        "expected_keywords": ["customer", "aws", "responsibility"],
        "expected_providers": ["aws"],
        "topic": "security",
        "must_contain": ["responsibility"],
        "must_not_contain": ["Azure policy"]
    }
]


azure_questions = [
    {
        "query": "How does Azure Virtual Network work?",
        "expected_keywords": ["subnet", "ip", "network"],
        "expected_providers": ["azure"],
        "topic": "networking",
        "must_contain": ["subnet"],
        "must_not_contain": ["AWS VPC"]
    },
    {
        "query": "Explain Azure RBAC",
        "expected_keywords": ["roles", "access", "permissions"],
        "expected_providers": ["azure"],
        "topic": "iam",
        "must_contain": ["roles"],
        "must_not_contain": ["IAM roles"]
    },
    {
        "query": "What are Azure management groups?",
        "expected_keywords": ["hierarchy", "subscription"],
        "expected_providers": ["azure"],
        "topic": "governance",
        "must_contain": ["subscriptions"],
        "must_not_contain": ["GCP folders"]
    },
    {
        "query": "Explain Azure landing zone",
        "expected_keywords": ["governance", "policy"],
        "expected_providers": ["azure"],
        "topic": "landing-zone",
        "must_contain": ["policy"],
        "must_not_contain": ["Control Tower"]
    },
    {
        "query": "What is Azure Policy?",
        "expected_keywords": ["compliance", "rules"],
        "expected_providers": ["azure"],
        "topic": "security",
        "must_contain": ["rules"],
        "must_not_contain": ["AWS Config"]
    }
]

gcp_questions = [
    {
        "query": "Explain GCP IAM roles",
        "expected_keywords": ["roles", "permissions", "policy"],
        "expected_providers": ["gcp"],
        "topic": "iam",
        "must_contain": ["roles"],
        "must_not_contain": ["AWS IAM"]
    },
    {
        "query": "What is GCP resource hierarchy?",
        "expected_keywords": ["organization", "folders", "projects"],
        "expected_providers": ["gcp"],
        "topic": "governance",
        "must_contain": ["projects"],
        "must_not_contain": ["subscriptions"]
    },
    {
        "query": "Explain VPC in GCP",
        "expected_keywords": ["network", "subnet"],
        "expected_providers": ["gcp"],
        "topic": "networking",
        "must_contain": ["subnet"],
        "must_not_contain": ["Azure VNet"]
    },
    {
        "query": "What is GCP organization policy?",
        "expected_keywords": ["constraints", "policy"],
        "expected_providers": ["gcp"],
        "topic": "security",
        "must_contain": ["policy"],
        "must_not_contain": ["AWS Config"]
    },
    {
        "query": "Explain GCP landing zone concept",
        "expected_keywords": ["projects", "structure"],
        "expected_providers": ["gcp"],
        "topic": "landing-zone",
        "must_contain": ["projects"],
        "must_not_contain": ["Control Tower"]
    }
]

multi_cloud_questions = [
    {
        "query": "Compare AWS IAM, Azure RBAC, and GCP IAM",
        "expected_keywords": ["roles", "permissions"],
        "expected_providers": ["aws", "azure", "gcp"],
        "topic": "iam",
        "must_contain": ["AWS", "Azure", "GCP"],
        "must_not_contain": []
    },
    {
        "query": "Compare AWS VPC, Azure VNet, and GCP VPC",
        "expected_keywords": ["network", "subnet"],
        "expected_providers": ["aws", "azure", "gcp"],
        "topic": "networking",
        "must_contain": ["VPC", "VNet"],
        "must_not_contain": []
    },
    {
        "query": "Compare landing zone concepts across AWS, Azure, GCP",
        "expected_keywords": ["governance"],
        "expected_providers": ["aws", "azure", "gcp"],
        "topic": "landing-zone",
        "must_contain": ["Control Tower", "management groups", "projects"],
        "must_not_contain": []
    },
    {
        "query": "Compare security models across AWS, Azure, and GCP",
        "expected_keywords": ["security"],
        "expected_providers": ["aws", "azure", "gcp"],
        "topic": "security",
        "must_contain": ["shared responsibility"],
        "must_not_contain": []
    },
    {
        "query": "Compare identity management across cloud providers",
        "expected_keywords": ["roles", "identity"],
        "expected_providers": ["aws", "azure", "gcp"],
        "topic": "iam",
        "must_contain": ["roles"],
        "must_not_contain": []
    }
]


edge_questions = [
    {
        "query": "What is Kubernetes pricing in AWS?",
        "expected_keywords": ["missing"],
        "expected_providers": ["aws"],
        "topic": "unknown",
        "must_contain": ["missing"],
        "must_not_contain": ["hallucinated pricing"]
    },
    {
        "query": "Does Azure VNet use IAM roles like AWS?",
        "expected_keywords": ["difference"],
        "expected_providers": ["azure", "aws"],
        "topic": "networking",
        "must_contain": ["difference"],
        "must_not_contain": []
    },
    {
        "query": "Explain quantum computing service in GCP",
        "expected_keywords": ["missing"],
        "expected_providers": ["gcp"],
        "topic": "unknown",
        "must_contain": ["missing"],
        "must_not_contain": ["fake explanation"]
    }
]



In [4]:
test_queries = (
    aws_questions +
    azure_questions +
    gcp_questions +
    multi_cloud_questions +
    edge_questions
)

### Heuristic Metrics

In [7]:
# retrieval_score_from_out
def retrieval_score_from_out(out):
    diag = out.get("retrieval", {})
    missing = diag.get("topic_missing_providers", [])
    requested = diag.get("requested_providers", [])

    if not requested:
        return 1.0

    return 1 - (len(missing) / len(requested))
    
# answer_correctness
def answer_correctness(answer, expected_keywords):
    answer = answer.lower()
    matches = sum(1 for kw in expected_keywords if kw in answer)
    return matches / len(expected_keywords)
    
# grounding_score_from_out
def grounding_score_from_out(out):
    g = out.get("grounding", {})
    return 1 if g.get("is_fully_grounded_for_requested", False) else 0
    
# provider_coverage
def provider_coverage(out):
    g = out.get("grounding", {})
    requested = set(g.get("requested", []))
    present = set(g.get("present", []))

    if not requested:
        return 1.0

    return len(present & requested) / len(requested)
    
# topic_cov_score_from_out
def topic_cov_score_from_out(out: dict) -> float:
    tc = out.get("topic_coverage") or {}
    req = tc.get("requested", [])
    miss = tc.get("topic_missing", [])
    if not req:
        return 1.0
    return max(0.0, 1.0 - (len(miss) / len(req)))

    
# semantic_grounding_score
def semantic_grounding_score(out):
    s = out.get("semantic_grounding", {})
    return 1 if s.get("is_semantically_grounded", False) else 0

### LLM Judge

In [8]:
# Requires OPENAI_API_KEY in .env
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def build_judge_prompt(question, context, answer):
    return f"""
You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system.

Your job is to evaluate the quality of the answer strictly based on the provided context.

---

Question:
{question}

---

Context:
{context}

---

Answer:
{answer}

---

Evaluate the answer on the following metrics (score from 1 to 5):

1. Correctness:
Is the answer factually correct based ONLY on the context?

2. Relevance:
Does the answer directly address the question?

3. Completeness:
Does the answer cover all important aspects of the question?

4. Groundedness:
Is the answer fully supported by the context (no hallucination)?

---

Return your evaluation in STRICT JSON format:

{{
    "correctness": <1-5>,
    "relevance": <1-5>,
    "completeness": <1-5>,
    "groundedness": <1-5>,
    "explanation": "<short explanation>"
}}
"""


def llm_judge(question, context, answer):
    prompt = build_judge_prompt(question, context, answer)

    response = judge_llm.invoke(prompt).content

    # ✅ REMOVE markdown ```json blocks
    response_clean = re.sub(r"```json|```", "", response).strip()

    try:
        result = json.loads(response_clean)
    except Exception as e:
        print("⚠️ JSON parsing failed")
        print("Raw:", response)
        print("Cleaned:", response_clean)
        return None

    return result

### Context Builder

In [9]:
def build_context_from_docs(docs, max_chars=4000):
    context = ""

    for d in docs:
        if d.page_content:
            context += d.page_content + "\n\n"

        if len(context) > max_chars:
            break

    return context

### Evaluation Loop

In [10]:
results = []

for test in test_queries:
    out = run_query(test["query"])

    answer = out.get("answer", "")
    docs = out.get("docs", [])

    context = build_context_from_docs(docs)

    judge_result = llm_judge(test["query"], context, answer)

    results.append({
        "query": test["query"],

        # Heuristic metrics
        "retrieval": retrieval_score_from_out(out),
        "answer_correctness": answer_correctness(answer, test["expected_keywords"]),
        "grounding": grounding_score_from_out(out),
        "provider_coverage": provider_coverage(out),
        "topic_coverage": topic_cov_score_from_out(out),
        "semantic_grounding": semantic_grounding_score(out),

        # LLM metrics
        "llm_correctness": judge_result["correctness"] if judge_result else None,
        "llm_relevance": judge_result["relevance"] if judge_result else None,
        "llm_completeness": judge_result["completeness"] if judge_result else None,
        "llm_groundedness": judge_result["groundedness"] if judge_result else None,
    })

INFO:chromadb.telemetry.product.posthog:Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
INFO:rag:[optionD:stage1_broad] Q='What is AWS Well Architected Framework?...' provider_counts={'aws': 8} total_docs=8
INFO:rag:[optionD:stage2_filtered(aws)] Q='What is AWS Well Architected Framework?...' provider_counts={'aws': 8} total_docs=8
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:rag:[optionD:stage1_broad] Q='Explain AWS IAM roles and policies...' provider_counts={'aws': 8} total_docs=8
INFO:rag:[optionD:stage2_filtered(aws)] Q='Explain AWS IAM roles and policies...' provider_counts={'aws': 

### Normalization

In [11]:
def normalize(score):
    return score / 5 if score else None

In [12]:
# DataFrame
df = pd.DataFrame(results)

for col in ["llm_correctness", "llm_relevance", "llm_completeness", "llm_groundedness"]:
    df[col] = df[col].apply(normalize)

df

,query,retrieval,answer_correctness,grounding,provider_coverage,topic_coverage,semantic_grounding,llm_correctness,llm_relevance,llm_completeness,llm_groundedness
0,What is AWS Well Architected Framework?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0
1,Explain AWS IAM roles and policies,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0
2,What is AWS Control Tower?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0
3,Explain AWS VPC components,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0
4,What is AWS shared responsibility model?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0
5,How does Azure Virtual Network work?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0
6,Explain Azure RBAC,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0
7,What are Azure management groups?,1.0,1.000000,1,1.0,0.0,1,1.0,1.0,1.0,1.0
8,Explain Azure landing zone,1.0,0.500000,1,1.0,1.0,1,1.0,1.0,1.0,1.0
9,What is Azure Policy?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0


In [13]:
def compute_final_score(row):
    return (
        0.3 * row["llm_correctness"] +
        0.2 * row["llm_completeness"] +
        0.2 * row["llm_relevance"] +
        0.2 * row["llm_groundedness"] +
        0.1 * row["retrieval"]
    )

In [14]:
df["final_score"] = df.apply(compute_final_score, axis=1)
df

,query,retrieval,answer_correctness,grounding,provider_coverage,topic_coverage,semantic_grounding,llm_correctness,llm_relevance,llm_completeness,llm_groundedness,final_score
0,What is AWS Well Architected Framework?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0,1.00
1,Explain AWS IAM roles and policies,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0,1.00
2,What is AWS Control Tower?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0,1.00
3,Explain AWS VPC components,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0,1.00
4,What is AWS shared responsibility model?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0,1.00
5,How does Azure Virtual Network work?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0,1.00
6,Explain Azure RBAC,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0,1.00
7,What are Azure management groups?,1.0,1.000000,1,1.0,0.0,1,1.0,1.0,1.0,1.0,1.00
8,Explain Azure landing zone,1.0,0.500000,1,1.0,1.0,1,1.0,1.0,1.0,1.0,1.00
9,What is Azure Policy?,1.0,1.000000,1,1.0,1.0,1,1.0,1.0,1.0,1.0,1.00


In [15]:
df["final_score"].mean()

0.8208695652173913

In [16]:
# save results
#df.to_excel('E://GenAi//project//Rag_cloud_platforms//rag_eval_runs//eval_results.xlsx')

## 🔍 Key Observations

- Topic-aware retrieval improves provider coverage
- Deterministic reranking improves relevance
- LLM judge helps validate answer quality